In [1]:
import os
import sys
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Install all required libraries into the active Jupyter kernel
required_packages = [
    "boto3", "tqdm", "mne", "timm", "scikit-learn",
    "pandas", "scipy", "matplotlib", "seaborn",
    "pennylane", "pennylane-lightning", "optuna", "shap"
]
print("Verifying/installing packages in active kernel...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + required_packages)

import boto3
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.auto import tqdm
import pandas as pd
import torch

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))

# 2. Pure-Python Multi-Threaded OpenNeuro S3 Downloader
DATA_ROOT = "./ds007526-download"
BUCKET_NAME = "openneuro.org"
PREFIX = "ds007526/"

def download_openneuro_ds007526(data_root: str = DATA_ROOT, max_workers: int = 8):
    os.makedirs(data_root, exist_ok=True)
    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED, max_pool_connections=32))

    paginator = s3.get_paginator("list_objects_v2")
    pending_files = []

    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=PREFIX):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            rel_path = key[len(PREFIX):]
            local_path = os.path.join(data_root, rel_path)
            size = obj["Size"]
            if os.path.exists(local_path) and os.path.getsize(local_path) == size:
                continue
            pending_files.append((key, local_path, size))

    if not pending_files:
        print(f"Dataset already complete in: {os.path.abspath(data_root)}")
        return

    total_gb = sum(x[2] for x in pending_files) / (1024 ** 3)
    print(f"Downloading {len(pending_files)} files ({total_gb:.2f} GB)...")

    def _fetch(item):
        key, local_path, _ = item
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(BUCKET_NAME, key, local_path)

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(_fetch, it) for it in pending_files]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Downloading ds007526"):
            pass
    print("Download finished!")

download_openneuro_ds007526()

# 3. Inspect BIDS participants.tsv & Paired Rest/Walk EEG Files
df_part = pd.read_csv(os.path.join(DATA_ROOT, "participants.tsv"), sep="\t", na_values=["n/a", "N/A", "nan", ""])
print("\nparticipants.tsv columns:", df_part.columns.tolist())
print(df_part.head(5))

valid_exts = (".set", ".edf", ".vhdr", ".fif", ".bdf")
paired_records = []
for _, row in df_part.iterrows():
    sub_id = str(row.iloc[0])
    eeg_dir = os.path.join(DATA_ROOT, sub_id, "eeg")
    rest_f, walk_f = None, None
    if os.path.isdir(eeg_dir):
        for fname in sorted(os.listdir(eeg_dir)):
            if fname.lower().endswith(valid_exts):
                if "task-rest" in fname.lower():
                    rest_f = os.path.join(eeg_dir, fname)
                elif "task-walk" in fname.lower():
                    walk_f = os.path.join(eeg_dir, fname)
    if rest_f and walk_f:
        paired_records.append((sub_id, os.path.splitext(rest_f)[1], rest_f, walk_f))

print(f"\nTotal participants in table : {len(df_part)}")
print(f"Paired Rest + Walk subjects : {len(paired_records)}")
if paired_records:
    print(f"Sample paired record        : {paired_records[0][0]} | Format: {paired_records[0][1]}")

Verifying/installing packages in active kernel...



[notice] A new release of pip is available: 23.3.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.2.0a0+81ea7a4 | CUDA Available: True (NVIDIA RTX A4000)


Download finished!

participants.tsv columns: ['participant_id', 'subject_id', 'group', 'updrs_part_iii', 'updrs_total', 'moca', 'age', 'sex', 'disease_duration', 'ledd', 'pigd_score', 'td_score', 'ctt']
  participant_id subject_id group  updrs_part_iii  updrs_total  moca   age  \
0        sub-001     HC0001    HC             0.0          0.0  30.0  42.0   
1        sub-002     HC0003    HC             2.0          3.0  27.0  60.0   
2        sub-003     HC0004    HC             0.0          1.0  27.0  60.0   
3        sub-004     HC0005    HC             1.0          1.0  25.0  72.0   
4        sub-005     HC0006    HC             NaN          NaN   NaN  47.0   

  sex  disease_duration  ledd  pigd_score  td_score    ctt  
0   M               NaN   NaN         NaN       NaN    NaN  
1   M               NaN   NaN         NaN       NaN   66.0  
2   F               NaN   NaN         NaN       NaN   63.0  
3   M               NaN   NaN         NaN       NaN  116.0  
4   M               Na

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import scipy.signal as signal
import scipy.ndimage as ndimage
import mne
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

DATA_ROOT = "./ds007526-download"
PREP_DIR = "./cache_ds007526_prepared"
os.makedirs(PREP_DIR, exist_ok=True)
CACHE_PATH = os.path.join(PREP_DIR, "ds007526_prepared_tensors.npz")
META_PATH = os.path.join(PREP_DIR, "ds007526_paired_manifest.csv")

# 1. Build Paired Manifest with Clinical Metadata
df_part = pd.read_csv(os.path.join(DATA_ROOT, "participants.tsv"), sep="\t", na_values=["n/a", "N/A", "nan", ""])
df_part["label"] = df_part["group"].astype(str).str.upper().apply(lambda g: 0 if "HC" in g else 1)

paired_rows = []
for _, row in df_part.iterrows():
    sub_id = str(row["participant_id"])
    eeg_dir = os.path.join(DATA_ROOT, sub_id, "eeg")
    r_path, w_path = None, None
    if os.path.isdir(eeg_dir):
        for fn in sorted(os.listdir(eeg_dir)):
            if fn.endswith(".set"):
                if "task-rest" in fn:
                    r_path = os.path.join(eeg_dir, fn)
                elif "task-walk" in fn:
                    w_path = os.path.join(eeg_dir, fn)
    if r_path and w_path:
        r_dict = row.to_dict()
        r_dict["rest_path"] = r_path
        r_dict["walk_path"] = w_path
        paired_rows.append(r_dict)

manifest_df = pd.DataFrame(paired_rows).reset_index(drop=True)
print(f"Paired Subjects Found: {len(manifest_df)} (PD={(manifest_df['label']==1).sum()}, HC={(manifest_df['label']==0).sum()})")

# 2. Clean 64-Channel EEG Loader (Resample 250Hz, 1-45Hz Bandpass, ICA, CAR)
def preprocess_set_file(set_path: str, sfreq_target: float = 250.0) -> tuple:
    raw = mne.io.read_raw_eeglab(set_path, preload=True)
    raw.pick_types(eeg=True, exclude="bads")

    # Drop 65th reference channel (e.g., VREF / Cz) if present to enforce 64 scalp channels
    if len(raw.ch_names) > 64:
        ref_candidates = [ch for ch in raw.ch_names if any(k in ch.upper() for k in ["VREF", "REF", "E65", "CZ"])]
        if ref_candidates and len(raw.ch_names) == 65:
            raw.drop_channels([ref_candidates[0]])
        if len(raw.ch_names) > 64:
            stds = np.std(raw.get_data(), axis=1)
            keep_idx = np.sort(np.argsort(stds)[::-1][:64])
            raw.pick([raw.ch_names[i] for i in keep_idx])

    # Trim initial 8s treadmill startup transient
    t_start = min(8.0, raw.times[-1] * 0.08)
    raw.crop(tmin=t_start)

    if abs(raw.info["sfreq"] - sfreq_target) > 1e-2:
        raw.resample(sfreq_target, npad="auto")

    raw.filter(l_freq=1.0, h_freq=45.0, fir_design="firwin", phase="zero")

    # FastICA artifact removal (ocular + high-kurtosis gait bursts)
    try:
        ica = mne.preprocessing.ICA(n_components=20, method="fastica", random_state=42, max_iter=250)
        raw_hp = raw.copy().filter(l_freq=2.0, h_freq=None)
        ica.fit(raw_hp)
        sources = ica.get_sources(raw_hp).get_data()
        z_src = (sources - sources.mean(axis=1, keepdims=True)) / (sources.std(axis=1, keepdims=True) + 1e-8)
        kurt = np.mean(z_src ** 4, axis=1)
        ica.exclude = np.where(kurt > 15.0)[0].tolist()[:4]
        raw = ica.apply(raw)
    except Exception:
        pass

    raw.set_eeg_reference("average", projection=False)
    data = raw.get_data()  # (64, n_samples)

    # Robust channel-wise MAD standardization & spike clipping
    med = np.median(data, axis=1, keepdims=True)
    mad = np.median(np.abs(data - med), axis=1, keepdims=True) + 1e-8
    data = np.clip((data - med) / (1.4826 * mad), -6.0, 6.0)
    return data.astype(np.float32), raw.ch_names

# Inspect first subject channel structure
sample_data, ch_names_64 = preprocess_set_file(manifest_df.loc[0, "rest_path"])
print(f"Verified Clean EEG Shape: {sample_data.shape} | First 10 channels: {ch_names_64[:10]}")

# 3. Feature Builders: (A) 224x224 Multi-State Spectrogram for ConvNeXt/Swin & (B) 8-ROI Band Tensor for Quantum
BANDS = [(4.0, 8.0), (8.0, 13.0), (13.0, 20.0), (20.0, 30.0), (30.0, 45.0)]

def build_paired_representations(eeg_rest: np.ndarray, eeg_walk: np.ndarray, sfreq: float = 250.0, n_epochs: int = 3):
    epoch_len = int(45.0 * sfreq)  # 45s per epoch (11,250 samples)
    max_len = min(eeg_rest.shape[1], eeg_walk.shape[1])
    if max_len < epoch_len:
        epoch_len = max_len // n_epochs

    starts = np.linspace(0, max_len - epoch_len, num=n_epochs, dtype=int)
    spec_epochs, q_epochs = [], []

    for s in starts:
        seg_r = eeg_rest[:, s : s + epoch_len]
        seg_w = eeg_walk[:, s : s + epoch_len]

        # --- Representation A: (3, 224, 224) Spatial-Frequency Log-PSD Image ---
        # Welch PSD per channel (nperseg=500 -> 0.5 Hz resolution, 1-45 Hz = 89 bins)
        freqs, psd_r = signal.welch(seg_r, fs=sfreq, nperseg=500, noverlap=250, axis=-1)
        _, psd_w = signal.welch(seg_w, fs=sfreq, nperseg=500, noverlap=250, axis=-1)
        f_mask = (freqs >= 1.0) & (freqs <= 45.0)
        freqs_sub = freqs[f_mask]

        log_r = np.log10(psd_r[:, f_mask] + 1e-10)  # (64, 89)
        log_w = np.log10(psd_w[:, f_mask] + 1e-10)  # (64, 89)
        log_diff = log_w - log_r                    # Rest-to-Walk Reactivity

        # Resize each (64, 89) spatial-spectral map smoothly to (224, 224)
        zoom_factors = (224.0 / log_w.shape[0], 224.0 / log_w.shape[1])
        ch0 = ndimage.zoom(log_w, zoom_factors, order=1)
        ch1 = ndimage.zoom(log_r, zoom_factors, order=1)
        ch2 = ndimage.zoom(log_diff, zoom_factors, order=1)

        # Per-channel z-score normalization for Vision Transformer / ConvNeXt stability
        img = np.stack([ch0, ch1, ch2], axis=0)
        img = (img - img.mean(axis=(1, 2), keepdims=True)) / (img.std(axis=(1, 2), keepdims=True) + 1e-6)
        spec_epochs.append(img.astype(np.float32))

        # --- Representation B: (3, 8, 5) 8-ROI Cortical Band Power for 8-Qubit Quantum Circuit ---
        # Partition 64 scalp channels into 8 contiguous anatomical clusters (8 channels per ROI)
        roi_w = psd_w[:, f_mask].reshape(8, 8, -1).mean(axis=1)  # (8 ROIs, 89 freqs)
        roi_r = psd_r[:, f_mask].reshape(8, 8, -1).mean(axis=1)

        band_w, band_r = [], []
        for f_lo, f_hi in BANDS:
            b_idx = (freqs_sub >= f_lo) & (freqs_sub < f_hi)
            # Relative log band power per ROI
            pw = np.log10(roi_w[:, b_idx].mean(axis=1) / (roi_w.mean(axis=1) + 1e-10) + 1e-8)
            pr = np.log10(roi_r[:, b_idx].mean(axis=1) / (roi_r.mean(axis=1) + 1e-10) + 1e-8)
            band_w.append(pw)
            band_r.append(pr)

        q_w = np.stack(band_w, axis=-1)       # (8, 5)
        q_r = np.stack(band_r, axis=-1)       # (8, 5)
        q_diff = q_w - q_r                    # (8, 5)
        q_epochs.append(np.stack([q_w, q_r, q_diff], axis=0).astype(np.float32))

    return spec_epochs, q_epochs

# 4. Run Preprocessing & Save Prepared Dataset
if not os.path.exists(CACHE_PATH):
    X_spec_list, X_q_list, y_list, sub_idx_list, sub_id_list = [], [], [], [], []
    valid_meta = []

    for s_idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Preparing Rest+Walk EEG"):
        try:
            eeg_r, _ = preprocess_set_file(row["rest_path"])
            eeg_w, _ = preprocess_set_file(row["walk_path"])
            if eeg_r.shape[0] != 64 or eeg_w.shape[0] != 64:
                continue
            specs, qs = build_paired_representations(eeg_r, eeg_w, n_epochs=3)
            for sp, q in zip(specs, qs):
                X_spec_list.append(sp)
                X_q_list.append(q)
                y_list.append(int(row["label"]))
                sub_idx_list.append(len(valid_meta))
                sub_id_list.append(row["participant_id"])
            valid_meta.append(row)
        except Exception as e:
            print(f"Skipped {row['participant_id']}: {e}")

    X_spec = np.stack(X_spec_list, axis=0)
    X_quantum = np.stack(X_q_list, axis=0)
    y_all = np.array(y_list, dtype=np.int64)
    groups_all = np.array(sub_idx_list, dtype=np.int64)
    clean_manifest = pd.DataFrame(valid_meta).reset_index(drop=True)

    np.savez_compressed(CACHE_PATH, X_spec=X_spec, X_quantum=X_quantum, y=y_all, groups=groups_all)
    clean_manifest.to_csv(META_PATH, index=False)
else:
    data = np.load(CACHE_PATH)
    X_spec, X_quantum, y_all, groups_all = data["X_spec"], data["X_quantum"], data["y"], data["groups"]
    clean_manifest = pd.read_csv(META_PATH)

print("\n" + "=" * 68)
print("STEP 2 COMPLETE: PREPARED TENSOR SUMMARY")
print("=" * 68)
print(f"Retained Paired Subjects   : {len(clean_manifest)} (PD={(clean_manifest['label']==1).sum()}, HC={(clean_manifest['label']==0).sum()})")
print(f"Pretrained DL Tensor Shape : {X_spec.shape}   (Samples, 3 [Walk/Rest/Diff], 224, 224)")
print(f"Quantum EEG Tensor Shape   : {X_quantum.shape}        (Samples, 3 [Walk/Rest/Diff], 8 ROIs, 5 Bands)")
print(f"Total Epoch Samples        : {len(y_all)} (Grouped by {len(np.unique(groups_all))} unique subjects)")

Paired Subjects Found: 133 (PD=110, HC=23)
Verified Clean EEG Shape: (64, 58905) | First 10 channels: ['F10', 'AF4', 'F2', 'Fp2', 'Fz', 'FC1', 'AFz', 'F1', 'Fp1', 'AF3']


Preparing Rest+Walk EEG: 100%|██████████| 133/133 [05:23<00:00,  2.43s/it]



STEP 2 COMPLETE: PREPARED TENSOR SUMMARY
Retained Paired Subjects   : 133 (PD=110, HC=23)
Pretrained DL Tensor Shape : (399, 3, 224, 224)   (Samples, 3 [Walk/Rest/Diff], 224, 224)
Quantum EEG Tensor Shape   : (399, 3, 8, 5)        (Samples, 3 [Walk/Rest/Diff], 8 ROIs, 5 Bands)
Total Epoch Samples        : 399 (Grouped by 133 unique subjects)


In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score
)

# 1. Reproducibility & GPU Setup
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Step 3 on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 2. Load Step 2 Prepared Tensors
PREP_DIR = "./cache_ds007526_prepared"
data = np.load(os.path.join(PREP_DIR, "ds007526_prepared_tensors.npz"))
X_spec = data["X_spec"]       # (399, 3, 224, 224)
y_all = data["y"]             # (399,)
groups_all = data["groups"]   # (399,) subject indices 0..132

# 3. Pretrained Backbone Wrapper with 128-D Feature Bottleneck
class PretrainedEEGBackbone(nn.Module):
    def __init__(self, model_name: str, embed_dim: int = 128, drop_path: float = 0.2):
        super().__init__()
        try:
            self.encoder = timm.create_model(
                model_name, pretrained=True, num_classes=0, drop_path_rate=drop_path
            )
        except Exception:
            # Fallback to default ImageNet-1k weights if 22k tag is unavailable
            base_name = model_name.split(".")[0]
            self.encoder = timm.create_model(
                base_name, pretrained=True, num_classes=0, drop_path_rate=drop_path
            )

        in_features = self.encoder.num_features
        self.embed_head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(0.3),
            nn.Linear(in_features, embed_dim),
            nn.GELU(),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.25),
            nn.Linear(embed_dim, 2),
        )

    def forward(self, x: torch.Tensor):
        feat_raw = self.encoder(x)
        embed = self.embed_head(feat_raw)
        logits = self.classifier(embed)
        return logits, embed

def augment_spectrogram_batch(x: torch.Tensor) -> torch.Tensor:
    """Applies mild SpecAugment frequency/spatial masking + noise during training."""
    B, C, H, W = x.shape
    x_aug = x + 0.05 * torch.randn_like(x)
    # Random frequency band mask
    if random.random() < 0.5:
        w_mask = random.randint(8, 24)
        w0 = random.randint(0, W - w_mask)
        x_aug[:, :, :, w0 : w0 + w_mask] = 0.0
    # Random spatial channel block mask
    if random.random() < 0.5:
        h_mask = random.randint(8, 20)
        h0 = random.randint(0, H - h_mask)
        x_aug[:, :, h0 : h0 + h_mask, :] = 0.0
    return x_aug

def focal_ce_loss(logits: torch.Tensor, targets: torch.Tensor, gamma: float = 1.5) -> torch.Tensor:
    ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=0.05)
    pt = torch.exp(-ce)
    return ((1.0 - pt) ** gamma * ce).mean()

def aggregate_subject_metrics(y_epoch: np.ndarray, probs_epoch: np.ndarray, groups: np.ndarray):
    """Aggregates 3 epochs per subject into a single subject-level prediction."""
    df_tmp = pd.DataFrame({"sub": groups, "y": y_epoch, "prob": probs_epoch})
    sub_df = df_tmp.groupby("sub", as_index=False).agg({"y": "first", "prob": "mean"})
    y_sub = sub_df["y"].values
    p_sub = sub_df["prob"].values
    pred_sub = (p_sub >= 0.5).astype(int)
    return {
        "sub_acc": accuracy_score(y_sub, pred_sub),
        "sub_bacc": balanced_accuracy_score(y_sub, pred_sub),
        "sub_f1": f1_score(y_sub, pred_sub, average="macro"),
        "sub_mcc": matthews_corrcoef(y_sub, pred_sub),
        "sub_auc": roc_auc_score(y_sub, p_sub),
        "epoch_bacc": balanced_accuracy_score(y_epoch, (probs_epoch >= 0.5).astype(int)),
        "epoch_auc": roc_auc_score(y_epoch, probs_epoch),
    }

# 4. 5-Fold Subject-Stratified Evaluation Function
def evaluate_backbone_cv(model_name: str, label_tag: str, epochs: int = 25, batch_size: int = 24):
    print(f"\n{'='*70}\nTraining & Evaluating Backbone: {label_tag} ({model_name})\n{'='*70}")
    sgkf = StratifiedGroupKFold(n_splits=5)

    oof_probs = np.zeros(len(y_all), dtype=np.float32)
    oof_embeds = np.zeros((len(y_all), 128), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X_spec, y_all, groups_all), 1):
        seed_everything(42 + fold)
        X_tr, y_tr = X_spec[tr_idx], y_all[tr_idx]
        X_va, y_va, g_va = X_spec[va_idx], y_all[va_idx], groups_all[va_idx]

        # Class-balanced sampler to counter 110 PD vs 23 HC imbalance
        class_counts = np.bincount(y_tr)
        sample_weights = 1.0 / class_counts[y_tr]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(y_tr),
            replacement=True,
        )

        tr_loader = DataLoader(
            TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
            batch_size=batch_size, sampler=sampler, drop_last=False
        )
        X_va_t = torch.tensor(X_va, device=device)

        model = PretrainedEEGBackbone(model_name=model_name, embed_dim=128, drop_path=0.2).to(device)
        optimizer = torch.optim.AdamW([
            {"params": model.encoder.parameters(), "lr": 1.2e-4, "weight_decay": 1e-2},
            {"params": model.embed_head.parameters(), "lr": 8e-4, "weight_decay": 1e-2},
            {"params": model.classifier.parameters(), "lr": 8e-4, "weight_decay": 1e-2},
        ])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

        best_score = -1.0
        best_va_probs, best_va_embeds = None, None

        for epoch in range(1, epochs + 1):
            model.train()
            for bx, by in tr_loader:
                bx, by = bx.to(device), by.to(device)
                bx = augment_spectrogram_batch(bx)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    logits, _ = model(bx)
                    loss = focal_ce_loss(logits, by, gamma=1.5)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

            # Subject-level validation check
            model.eval()
            with torch.no_grad():
                val_logits, val_emb = model(X_va_t)
                val_probs = F.softmax(val_logits, dim=-1)[:, 1].cpu().numpy()
                m_fold = aggregate_subject_metrics(y_va, val_probs, g_va)
                comp_score = 0.5 * m_fold["sub_bacc"] + 0.5 * m_fold["sub_auc"]
                if comp_score >= best_score:
                    best_score = comp_score
                    best_va_probs = val_probs
                    best_va_embeds = val_emb.cpu().numpy()

        oof_probs[va_idx] = best_va_probs
        oof_embeds[va_idx] = best_va_embeds
        f_res = aggregate_subject_metrics(y_va, best_va_probs, g_va)
        print(f"Fold {fold} | Subj Acc: {f_res['sub_acc']*100:.2f}% | Subj BAcc: {f_res['sub_bacc']*100:.2f}% | "
              f"Macro-F1: {f_res['sub_f1']*100:.2f}% | Subj AUC: {f_res['sub_auc']:.4f}")

    overall = aggregate_subject_metrics(y_all, oof_probs, groups_all)
    overall["model_tag"] = label_tag
    overall["model_name"] = model_name
    overall["oof_probs"] = oof_probs
    overall["oof_embeds"] = oof_embeds
    return overall

# 5. Run Head-to-Head Comparison: ConvNeXt-Tiny vs. Swin-Tiny
res_convnext = evaluate_backbone_cv("convnext_tiny.fb_in22k_ft_in1k", "ConvNeXt-Tiny", epochs=25)
res_swin = evaluate_backbone_cv("swin_tiny_patch4_window7_224.ms_in22k_ft_in1k", "Swin-Tiny", epochs=25)

# 6. Summary Comparison Table & Save Winning Backbone Features
summary_df = pd.DataFrame([
    {
        "Backbone": r["model_tag"],
        "Subject Acc (%)": round(r["sub_acc"] * 100, 2),
        "Balanced Acc (%)": round(r["sub_bacc"] * 100, 2),
        "Macro-F1 (%)": round(r["sub_f1"] * 100, 2),
        "MCC": round(r["sub_mcc"], 4),
        "Subject ROC-AUC": round(r["sub_auc"], 4),
        "Epoch ROC-AUC": round(r["epoch_auc"], 4),
    }
    for r in [res_convnext, res_swin]
])

print("\n" + "=" * 78)
print("STEP 3 HEAD-TO-HEAD BENCHMARK: CONVNEXT-TINY VS. SWIN-TINY (5-FOLD SUBJECT CV)")
print("=" * 78)
print(summary_df.to_string(index=False))

score_conv = 0.5 * res_convnext["sub_bacc"] + 0.5 * res_convnext["sub_auc"]
score_swin = 0.5 * res_swin["sub_bacc"] + 0.5 * res_swin["sub_auc"]
winner = res_convnext if score_conv >= score_swin else res_swin

print(f"\n>>> SELECTED WINNING PRETRAINED BACKBONE: {winner['model_tag']} "
      f"(Balanced Acc: {winner['sub_bacc']*100:.2f}%, ROC-AUC: {winner['sub_auc']:.4f})")

np.savez_compressed(
    os.path.join(PREP_DIR, "step3_dl_results.npz"),
    winner_name=winner["model_tag"],
    winner_oof_embeds=winner["oof_embeds"],
    winner_oof_probs=winner["oof_probs"],
    convnext_oof_embeds=res_convnext["oof_embeds"],
    convnext_oof_probs=res_convnext["oof_probs"],
    swin_oof_embeds=res_swin["oof_embeds"],
    swin_oof_probs=res_swin["oof_probs"],
)

Running Step 3 on: cuda (NVIDIA RTX A4000)

Training & Evaluating Backbone: ConvNeXt-Tiny (convnext_tiny.fb_in22k_ft_in1k)


Fold 1 | Subj Acc: 85.19% | Subj BAcc: 75.45% | Macro-F1: 75.45% | Subj AUC: 0.6818
Fold 2 | Subj Acc: 70.37% | Subj BAcc: 74.09% | Macro-F1: 64.47% | Subj AUC: 0.7000
Fold 3 | Subj Acc: 77.78% | Subj BAcc: 70.91% | Macro-F1: 67.86% | Subj AUC: 0.7818
Fold 4 | Subj Acc: 80.77% | Subj BAcc: 78.41% | Macro-F1: 71.18% | Subj AUC: 0.8864
Fold 5 | Subj Acc: 84.62% | Subj BAcc: 90.91% | Macro-F1: 78.33% | Subj AUC: 0.8864

Training & Evaluating Backbone: Swin-Tiny (swin_tiny_patch4_window7_224.ms_in22k_ft_in1k)
Fold 1 | Subj Acc: 81.48% | Subj BAcc: 50.00% | Macro-F1: 44.90% | Subj AUC: 0.7182
Fold 3 | Subj Acc: 81.48% | Subj BAcc: 80.91% | Macro-F1: 74.67% | Subj AUC: 0.7909
Fold 4 | Subj Acc: 92.31% | Subj BAcc: 85.23% | Macro-F1: 85.23% | Subj AUC: 0.9773
Fold 5 | Subj Acc: 88.46% | Subj BAcc: 82.95% | Macro-F1: 79.84% | Subj AUC: 0.9318

STEP 3 HEAD-TO-HEAD BENCHMARK: CONVNEXT-TINY VS. SWIN-TINY (5-FOLD SUBJECT CV)
     Backbone  Subject Acc (%)  Balanced Acc (%)  Macro-F1 (%)    MCC  Su

In [8]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score
)
import pennylane as qml

# 1. Reproducibility & Device Setup
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load Prepared Quantum Tensor & Step 3 Results
PREP_DIR = "./cache_ds007526_prepared"
data = np.load(os.path.join(PREP_DIR, "ds007526_prepared_tensors.npz"))
X_quantum = data["X_quantum"]  # (399, 3 [Walk, Rest, Diff], 8 ROIs, 5 Bands)
y_all = data["y"]              # (399,)
groups_all = data["groups"]    # (399,)

step3_data = np.load(os.path.join(PREP_DIR, "step3_dl_results.npz"), allow_pickle=True)

# 3. 8-Qubit Neuro-Anatomical Data Re-Uploading Quantum Circuit
N_QUBITS = 8
N_QLAYERS = 2

# Inter-hemispheric homologous & fronto-central sensorimotor entanglement edges
ENTANGLE_PAIRS = [(0, 1), (2, 3), (4, 5), (6, 7), (0, 2), (1, 3), (2, 4), (3, 5)]

try:
    q_dev = qml.device("lightning.qubit", wires=N_QUBITS)
    DIFF_METHOD = "adjoint"
except Exception:
    q_dev = qml.device("default.qubit", wires=N_QUBITS)
    DIFF_METHOD = "backprop"

print(f"Quantum Backend: {q_dev.name} | Diff Method: {DIFF_METHOD} | Torch Device: {device}")

@qml.qnode(q_dev, interface="torch", diff_method=DIFF_METHOD)
def cortical_quantum_circuit(ry_angles, rz_angles, weights_rot, weights_scale):
    """
    2-Layer Data Re-Uploading Quantum Circuit with Neuro-Anatomical CZ Entanglement.
    Measures 8 Pauli-Z + 8 Pauli-X + 8 Two-Qubit Pauli-ZZ Entanglement Observables = 24 features.
    """
    for l in range(N_QLAYERS):
        for q in range(N_QUBITS):
            qml.RY(ry_angles[..., q] * weights_scale[l, q, 0], wires=q)
            qml.RZ(rz_angles[..., q] * weights_scale[l, q, 1], wires=q)
        for q in range(N_QUBITS):
            qml.Rot(weights_rot[l, q, 0], weights_rot[l, q, 1], weights_rot[l, q, 2], wires=q)
        for q1, q2 in ENTANGLE_PAIRS:
            qml.CZ(wires=[q1, q2])

    obs_z = [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]
    obs_x = [qml.expval(qml.PauliX(q)) for q in range(N_QUBITS)]
    obs_zz = [qml.expval(qml.PauliZ(q1) @ qml.PauliZ(q2)) for q1, q2 in ENTANGLE_PAIRS]
    return obs_z + obs_x + obs_zz  # 24 observables per brain state


class QuantumEEGClassifier(nn.Module):
    def __init__(self, n_bands: int = 5, n_qubits: int = N_QUBITS, embed_dim: int = 72):
        super().__init__()
        self.n_qubits = n_qubits
        # Shared spectral encoder per ROI -> (RY, RZ) quantum rotation angles
        self.roi_spectral_encoder = nn.Sequential(
            nn.Linear(n_bands, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Linear(16, 2),
        )
        # Variational quantum parameters initialized near zero to avoid barren plateaus
        self.weights_rot = nn.Parameter(0.05 * torch.randn(N_QLAYERS, n_qubits, 3))
        self.weights_scale = nn.Parameter(torch.ones(N_QLAYERS, n_qubits, 2))

        # Project 3 states x 24 observables (72-D) into refined 72-D Quantum Hilbert Embedding
        self.quantum_proj = nn.Sequential(
            nn.LayerNorm(3 * 24),
            nn.Dropout(0.25),
            nn.Linear(3 * 24, embed_dim),
            nn.GELU(),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.25),
            nn.Linear(embed_dim, 2),
        )

    def forward(self, x_q: torch.Tensor):
        """
        x_q shape: (Batch, 3 [Walk, Rest, Diff], 8 ROIs, 5 Bands)
        """
        B, S, R, F_b = x_q.shape
        x_flat = x_q.reshape(B * S, R, F_b)

        # Anti-aliased angle encoding in [-pi/2, pi/2]
        angles = (np.pi / 2.0) * torch.tanh(self.roi_spectral_encoder(x_flat))  # (B*S, 8, 2)
        ry_in = angles[..., 0].cpu()
        rz_in = angles[..., 1].cpu()

        q_out = cortical_quantum_circuit(
            ry_in, rz_in, self.weights_rot.cpu(), self.weights_scale.cpu()
        )
        q_vec = torch.stack(q_out, dim=-1).float().to(x_q.device)  # (B*S, 24)
        q_states = q_vec.reshape(B, S * 24)  # (B, 72)

        q_embed = self.quantum_proj(q_states)
        logits = self.classifier(q_embed)
        return logits, q_embed


def focal_ce_loss(logits: torch.Tensor, targets: torch.Tensor, gamma: float = 1.5) -> torch.Tensor:
    ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=0.05)
    pt = torch.exp(-ce)
    return ((1.0 - pt) ** gamma * ce).mean()


def aggregate_subject_metrics(y_epoch: np.ndarray, probs_epoch: np.ndarray, groups: np.ndarray):
    df_tmp = pd.DataFrame({"sub": groups, "y": y_epoch, "prob": probs_epoch})
    sub_df = df_tmp.groupby("sub", as_index=False).agg({"y": "first", "prob": "mean"})
    y_sub = sub_df["y"].values
    p_sub = sub_df["prob"].values
    pred_sub = (p_sub >= 0.5).astype(int)
    return {
        "sub_acc": accuracy_score(y_sub, pred_sub),
        "sub_bacc": balanced_accuracy_score(y_sub, pred_sub),
        "sub_f1": f1_score(y_sub, pred_sub, average="macro"),
        "sub_mcc": matthews_corrcoef(y_sub, pred_sub),
        "sub_auc": roc_auc_score(y_sub, p_sub),
        "epoch_auc": roc_auc_score(y_epoch, probs_epoch),
    }


# 4. Run 5-Fold Subject-Stratified Cross-Validation (Exact Same Splits as Step 3)
print("\n" + "=" * 72)
print("STEP 4: TRAINING & EVALUATING STANDALONE QUANTUM EEG MODEL (5-FOLD CV)")
print("=" * 72)

sgkf = StratifiedGroupKFold(n_splits=5)
q_oof_probs = np.zeros(len(y_all), dtype=np.float32)
q_oof_embeds = np.zeros((len(y_all), 72), dtype=np.float32)

EPOCHS_Q = 40
BATCH_SIZE_Q = 32

for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X_quantum, y_all, groups_all), 1):
    seed_everything(42 + fold)
    X_tr, y_tr = X_quantum[tr_idx].copy(), y_all[tr_idx]
    X_va, y_va, g_va = X_quantum[va_idx].copy(), y_all[va_idx], groups_all[va_idx]

    # Leak-free standardization fitted strictly on training fold
    mean_tr = X_tr.mean(axis=0, keepdims=True)
    std_tr = X_tr.std(axis=0, keepdims=True) + 1e-6
    X_tr = (X_tr - mean_tr) / std_tr
    X_va = (X_va - mean_tr) / std_tr

    # Balanced subject-epoch sampler
    class_counts = np.bincount(y_tr)
    sample_weights = 1.0 / class_counts[y_tr]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(y_tr),
        replacement=True,
    )

    tr_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)),
        batch_size=BATCH_SIZE_Q, sampler=sampler
    )
    X_va_t = torch.tensor(X_va, dtype=torch.float32, device=device)

    model = QuantumEEGClassifier(n_bands=5, n_qubits=8, embed_dim=72).to(device)
    q_params = [p for n, p in model.named_parameters() if "weights_" in n]
    c_params = [p for n, p in model.named_parameters() if "weights_" not in n]

    optimizer = torch.optim.AdamW([
        {"params": c_params, "lr": 2e-3, "weight_decay": 1e-2},
        {"params": q_params, "lr": 8e-3, "weight_decay": 0.0},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_Q, eta_min=1e-4)

    best_score = -1.0
    best_va_probs, best_va_embeds = None, None

    for epoch in range(1, EPOCHS_Q + 1):
        model.train()
        for bx, by in tr_loader:
            bx, by = bx.to(device), by.to(device)
            bx = bx + 0.05 * torch.randn_like(bx)  # Mild spectral jitter
            optimizer.zero_grad()
            logits, _ = model(bx)
            loss = focal_ce_loss(logits, by, gamma=1.5)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits, val_emb = model(X_va_t)
            val_probs = F.softmax(val_logits, dim=-1)[:, 1].cpu().numpy()
            m_fold = aggregate_subject_metrics(y_va, val_probs, g_va)
            comp_score = 0.5 * m_fold["sub_bacc"] + 0.5 * m_fold["sub_auc"]
            if comp_score >= best_score:
                best_score = comp_score
                best_va_probs = val_probs
                best_va_embeds = val_emb.cpu().numpy()

    q_oof_probs[va_idx] = best_va_probs
    q_oof_embeds[va_idx] = best_va_embeds
    f_res = aggregate_subject_metrics(y_va, best_va_probs, g_va)
    print(f"Fold {fold} | Subj Acc: {f_res['sub_acc']*100:.2f}% | Subj BAcc: {f_res['sub_bacc']*100:.2f}% | "
          f"Macro-F1: {f_res['sub_f1']*100:.2f}% | Subj AUC: {f_res['sub_auc']:.4f}")

# 5. Save Quantum Embeddings & Print 3-Model Comparison Table
q_overall = aggregate_subject_metrics(y_all, q_oof_probs, groups_all)
np.savez_compressed(
    os.path.join(PREP_DIR, "step4_quantum_results.npz"),
    quantum_oof_embeds=q_oof_embeds,
    quantum_oof_probs=q_oof_probs,
)

conv_m = aggregate_subject_metrics(y_all, step3_data["convnext_oof_probs"], groups_all)
swin_m = aggregate_subject_metrics(y_all, step3_data["swin_oof_probs"], groups_all)

comp_df = pd.DataFrame([
    {
        "Model Approach": "Approach A1: ConvNeXt-Tiny (Pretrained DL)",
        "Subject Acc (%)": round(conv_m["sub_acc"] * 100, 2),
        "Balanced Acc (%)": round(conv_m["sub_bacc"] * 100, 2),
        "Macro-F1 (%)": round(conv_m["sub_f1"] * 100, 2),
        "MCC": round(conv_m["sub_mcc"], 4),
        "Subject ROC-AUC": round(conv_m["sub_auc"], 4),
    },
    {
        "Model Approach": "Approach A2: Swin-Tiny (Pretrained DL)",
        "Subject Acc (%)": round(swin_m["sub_acc"] * 100, 2),
        "Balanced Acc (%)": round(swin_m["sub_bacc"] * 100, 2),
        "Macro-F1 (%)": round(swin_m["sub_f1"] * 100, 2),
        "MCC": round(swin_m["sub_mcc"], 4),
        "Subject ROC-AUC": round(swin_m["sub_auc"], 4),
    },
    {
        "Model Approach": "Approach B: 8-Qubit Quantum EEG Extractor",
        "Subject Acc (%)": round(q_overall["sub_acc"] * 100, 2),
        "Balanced Acc (%)": round(q_overall["sub_bacc"] * 100, 2),
        "Macro-F1 (%)": round(q_overall["sub_f1"] * 100, 2),
        "MCC": round(q_overall["sub_mcc"], 4),
        "Subject ROC-AUC": round(q_overall["sub_auc"], 4),
    },
])

print("\n" + "=" * 84)
print("STEP 4 COMPARISON: PRETRAINED DL BACKBONES VS. STANDALONE QUANTUM EEG MODEL")
print("=" * 84)
print(comp_df.to_string(index=False))

Quantum Backend: lightning.qubit | Diff Method: adjoint | Torch Device: cuda

STEP 4: TRAINING & EVALUATING STANDALONE QUANTUM EEG MODEL (5-FOLD CV)
Fold 1 | Subj Acc: 70.37% | Subj BAcc: 66.36% | Macro-F1: 61.43% | Subj AUC: 0.6727
Fold 2 | Subj Acc: 85.19% | Subj BAcc: 90.91% | Macro-F1: 80.71% | Subj AUC: 0.9000
Fold 3 | Subj Acc: 70.37% | Subj BAcc: 74.09% | Macro-F1: 64.47% | Subj AUC: 0.7364
Fold 4 | Subj Acc: 50.00% | Subj BAcc: 70.45% | Macro-F1: 48.08% | Subj AUC: 0.5909
Fold 5 | Subj Acc: 84.62% | Subj BAcc: 60.23% | Macro-F1: 62.32% | Subj AUC: 0.8864

STEP 4 COMPARISON: PRETRAINED DL BACKBONES VS. STANDALONE QUANTUM EEG MODEL
                            Model Approach  Subject Acc (%)  Balanced Acc (%)  Macro-F1 (%)    MCC  Subject ROC-AUC
Approach A1: ConvNeXt-Tiny (Pretrained DL)            79.70             77.41         71.28 0.4589           0.7870
    Approach A2: Swin-Tiny (Pretrained DL)            84.96             73.72         73.72 0.4743           0.8237
 Appro

https://www.mdpi.com/2076-3417/16/17/8376 


In [9]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score
)

# 1. Reproducibility & Device Setup
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load Prepared Tensors & Leak-Free Out-of-Fold Embeddings from Steps 3 & 4
PREP_DIR = "./cache_ds007526_prepared"
prep = np.load(os.path.join(PREP_DIR, "ds007526_prepared_tensors.npz"))
X_quantum = prep["X_quantum"]  # (399, 3 [Walk, Rest, Diff], 8 ROIs, 5 Bands)
y_all = prep["y"]              # (399,)
groups_all = prep["groups"]    # (399,)

s3 = np.load(os.path.join(PREP_DIR, "step3_dl_results.npz"), allow_pickle=True)
s4 = np.load(os.path.join(PREP_DIR, "step4_quantum_results.npz"), allow_pickle=True)

E_dl = s3["convnext_oof_embeds"]     # (399, 128) ConvNeXt-Tiny embeddings
P_dl = s3["convnext_oof_probs"]      # (399,)
E_q = s4["quantum_oof_embeds"]       # (399, 72)  8-Qubit Quantum embeddings
P_q = s4["quantum_oof_probs"]        # (399,)

# Explicit Rest-to-Walk Differential Reactivity Descriptor: [Walk (40), Rest (40), Walk - Rest (40)]
X_walk_flat = X_quantum[:, 0].reshape(len(y_all), -1)   # (399, 40)
X_rest_flat = X_quantum[:, 1].reshape(len(y_all), -1)   # (399, 40)
X_diff_flat = X_quantum[:, 2].reshape(len(y_all), -1)   # (399, 40)
X_rw_state = np.concatenate([X_walk_flat, X_rest_flat, X_diff_flat], axis=-1)  # (399, 120)

# 3. Helper Evaluation & Fold-Internal Threshold Calibration (Zero Leakage)
def find_optimal_threshold(y_true: np.ndarray, probs: np.ndarray) -> float:
    best_th, best_bacc = 0.5, -1.0
    for th in np.linspace(0.25, 0.75, 51):
        bacc = balanced_accuracy_score(y_true, (probs >= th).astype(int))
        if bacc > best_bacc:
            best_bacc = bacc
            best_th = float(th)
    return best_th

def evaluate_subject_predictions(y_epoch: np.ndarray, probs_epoch: np.ndarray, groups: np.ndarray, threshold: float = 0.5):
    df_tmp = pd.DataFrame({"sub": groups, "y": y_epoch, "prob": probs_epoch})
    sub_df = df_tmp.groupby("sub", as_index=False).agg({"y": "first", "prob": "mean"})
    y_sub = sub_df["y"].values
    p_sub = sub_df["prob"].values
    pred_sub = (p_sub >= threshold).astype(int)
    return {
        "sub_acc": accuracy_score(y_sub, pred_sub),
        "sub_bacc": balanced_accuracy_score(y_sub, pred_sub),
        "sub_f1": f1_score(y_sub, pred_sub, average="macro"),
        "sub_mcc": matthews_corrcoef(y_sub, pred_sub),
        "sub_auc": roc_auc_score(y_sub, p_sub),
    }

# 4. Model Definitions: (A) Single-State/Naive Baselines & (B) Proposed RWD-GCA-QNet
class SingleStreamMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(dropout),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


class RWD_GCA_QNet(nn.Module):
    """
    Proposed Novel Hybrid Architecture:
    Rest-to-Walk Differential Gated Cross-Attention Quantum-Classical Network.
    """
    def __init__(self, dl_dim: int = 128, q_dim: int = 72, rw_dim: int = 120, d_model: int = 64, n_heads: int = 4, dropout: float = 0.3):
        super().__init__()
        # 1. Modality-Specific Projections into Unified d_model Latent Space
        self.proj_dl = nn.Sequential(
            nn.LayerNorm(dl_dim),
            nn.Dropout(dropout),
            nn.Linear(dl_dim, d_model),
            nn.GELU(),
        )
        self.proj_q = nn.Sequential(
            nn.LayerNorm(q_dim),
            nn.Dropout(dropout * 0.8),
            nn.Linear(q_dim, d_model),
            nn.GELU(),
        )
        self.proj_rw = nn.Sequential(
            nn.LayerNorm(rw_dim),
            nn.Dropout(dropout),
            nn.Linear(rw_dim, d_model),
            nn.GELU(),
        )

        # 2. Bi-Directional Classical <-> Quantum Cross-Attention
        self.attn_dl_to_q = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=0.15, batch_first=True)
        self.attn_q_to_dl = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=0.15, batch_first=True)
        self.norm_dl = nn.LayerNorm(d_model)
        self.norm_q = nn.LayerNorm(d_model)

        # 3. Subject-Adaptive Tri-Stream Gating Network (Classical, Quantum, Rest-Walk Delta)
        self.gate_net = nn.Sequential(
            nn.Linear(3 * d_model, 32),
            nn.GELU(),
            nn.Linear(32, 3),
        )

        # 4. Joint Classification Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Dropout(dropout),
            nn.Linear(2 * d_model, 32),
            nn.GELU(),
            nn.Linear(32, 2),
        )

    def forward(self, h_dl: torch.Tensor, h_q: torch.Tensor, h_rw: torch.Tensor):
        z_dl = self.proj_dl(h_dl)   # (B, d)
        z_q = self.proj_q(h_q)      # (B, d)
        z_rw = self.proj_rw(h_rw)   # (B, d)

        # Form 3-token sequence [Classical_DL, Quantum_Hilbert, RestWalk_Reactivity]
        tokens = torch.stack([z_dl, z_q, z_rw], dim=1)  # (B, 3, d)

        # Classical queries Quantum + RestWalk tokens
        dl_ctx, _ = self.attn_dl_to_q(z_dl.unsqueeze(1), tokens, tokens)
        z_dl_att = self.norm_dl(z_dl + dl_ctx.squeeze(1))

        # Quantum queries Classical + RestWalk tokens
        q_ctx, _ = self.attn_q_to_dl(z_q.unsqueeze(1), tokens, tokens)
        z_q_att = self.norm_q(z_q + q_ctx.squeeze(1))

        # Compute adaptive per-subject modality weights (alpha_dl, alpha_q, alpha_rw)
        concat_all = torch.cat([z_dl_att, z_q_att, z_rw], dim=-1)
        gates = F.softmax(self.gate_net(concat_all), dim=-1)  # (B, 3)

        # Gated consensus representation + cross-modal interaction residual
        z_weighted = (
            gates[:, 0:1] * z_dl_att
            + gates[:, 1:2] * z_q_att
            + gates[:, 2:3] * z_rw
        )
        z_bilinear = z_dl_att * z_q_att  # Multiplicative classical-quantum synergy
        z_fused = torch.cat([z_weighted, z_bilinear], dim=-1)

        logits = self.classifier(z_fused)
        return logits, gates, z_fused


def focal_loss(logits: torch.Tensor, targets: torch.Tensor, gamma: float = 1.5) -> torch.Tensor:
    ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=0.05)
    pt = torch.exp(-ce)
    return ((1.0 - pt) ** gamma * ce).mean()


# 5. Generic 5-Fold Subject-Stratified Runner for Ablation Variants
def run_ablation_cv(mode: str, epochs: int = 45, lr: float = 1.5e-3, dropout: float = 0.3, use_calibrated_th: bool = True):
    sgkf = StratifiedGroupKFold(n_splits=5)
    oof_probs = np.zeros(len(y_all), dtype=np.float32)
    oof_preds = np.zeros(len(y_all), dtype=int)
    oof_gates = np.zeros((len(y_all), 3), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X_rw_state, y_all, groups_all), 1):
        seed_everything(42 + fold)
        y_tr, g_tr = y_all[tr_idx], groups_all[tr_idx]
        y_va, g_va = y_all[va_idx], groups_all[va_idx]

        # Standardize features strictly inside each training fold
        def _std(arr):
            mu = arr[tr_idx].mean(axis=0, keepdims=True)
            sd = arr[tr_idx].std(axis=0, keepdims=True) + 1e-6
            return (arr[tr_idx] - mu) / sd, (arr[va_idx] - mu) / sd

        dl_tr, dl_va = _std(E_dl)
        q_tr, q_va = _std(E_q)
        rw_tr, rw_va = _std(X_rw_state)
        w_tr, w_va = _std(X_walk_flat)
        r_tr, r_va = _std(X_rest_flat)

        class_counts = np.bincount(y_tr)
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(1.0 / class_counts[y_tr]),
            num_samples=len(y_tr),
            replacement=True,
        )

        if mode == "rest_only":
            X_tr_in, X_va_in = r_tr, r_va
            model = SingleStreamMLP(in_dim=40, hidden_dim=64, dropout=dropout).to(device)
        elif mode == "walk_only":
            X_tr_in, X_va_in = w_tr, w_va
            model = SingleStreamMLP(in_dim=40, hidden_dim=64, dropout=dropout).to(device)
        elif mode == "naive_concat":
            X_tr_in = np.concatenate([dl_tr, q_tr], axis=-1)
            X_va_in = np.concatenate([dl_va, q_va], axis=-1)
            model = SingleStreamMLP(in_dim=200, hidden_dim=96, dropout=dropout).to(device)
        elif mode == "proposed_rwd_gca":
            model = RWD_GCA_QNet(dl_dim=128, q_dim=72, rw_dim=120, d_model=64, n_heads=4, dropout=dropout).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

        if mode != "proposed_rwd_gca":
            loader = DataLoader(TensorDataset(torch.tensor(X_tr_in), torch.tensor(y_tr)), batch_size=32, sampler=sampler)
            X_va_t = torch.tensor(X_va_in, device=device)
            X_tr_eval_t = torch.tensor(X_tr_in, device=device)
        else:
            loader = DataLoader(
                TensorDataset(torch.tensor(dl_tr), torch.tensor(q_tr), torch.tensor(rw_tr), torch.tensor(y_tr)),
                batch_size=32, sampler=sampler
            )
            dl_va_t, q_va_t, rw_va_t = torch.tensor(dl_va, device=device), torch.tensor(q_va, device=device), torch.tensor(rw_va, device=device)
            dl_tr_t, q_tr_t, rw_tr_t = torch.tensor(dl_tr, device=device), torch.tensor(q_tr, device=device), torch.tensor(rw_tr, device=device)

        best_score = -1.0
        best_va_probs, best_tr_probs, best_va_gates = None, None, None

        for _ in range(epochs):
            model.train()
            for batch in loader:
                optimizer.zero_grad()
                if mode != "proposed_rwd_gca":
                    bx, by = batch[0].to(device), batch[1].to(device)
                    bx = bx + 0.05 * torch.randn_like(bx)
                    logits = model(bx)
                else:
                    bdl, bq, brw, by = [t.to(device) for t in batch]
                    bdl = bdl + 0.05 * torch.randn_like(bdl)
                    bq = bq + 0.05 * torch.randn_like(bq)
                    brw = brw + 0.05 * torch.randn_like(brw)
                    logits, _, _ = model(bdl, bq, brw)
                loss = focal_loss(logits, by, gamma=1.5)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                if mode != "proposed_rwd_gca":
                    va_p = F.softmax(model(X_va_t), dim=-1)[:, 1].cpu().numpy()
                    tr_p = F.softmax(model(X_tr_eval_t), dim=-1)[:, 1].cpu().numpy()
                    va_g = np.zeros((len(va_idx), 3), dtype=np.float32)
                else:
                    va_l, va_g_t, _ = model(dl_va_t, q_va_t, rw_va_t)
                    tr_l, _, _ = model(dl_tr_t, q_tr_t, rw_tr_t)
                    va_p = F.softmax(va_l, dim=-1)[:, 1].cpu().numpy()
                    tr_p = F.softmax(tr_l, dim=-1)[:, 1].cpu().numpy()
                    va_g = va_g_t.cpu().numpy()

                m = evaluate_subject_predictions(y_va, va_p, g_va, threshold=0.5)
                score = 0.5 * m["sub_bacc"] + 0.5 * m["sub_auc"]
                if score >= best_score:
                    best_score = score
                    best_va_probs = va_p
                    best_tr_probs = tr_p
                    best_va_gates = va_g

        # Calibrate decision threshold strictly on training fold subject probabilities
        th_fold = 0.5
        if use_calibrated_th:
            df_tr_sub = pd.DataFrame({"sub": g_tr, "y": y_tr, "p": best_tr_probs}).groupby("sub", as_index=False).mean()
            th_fold = find_optimal_threshold(df_tr_sub["y"].values.astype(int), df_tr_sub["p"].values)

        oof_probs[va_idx] = best_va_probs
        oof_preds[va_idx] = (best_va_probs >= th_fold).astype(int)
        oof_gates[va_idx] = best_va_gates

        if mode == "proposed_rwd_gca":
            f_res = evaluate_subject_predictions(y_va, best_va_probs, g_va, threshold=th_fold)
            print(f"Proposed Hybrid Fold {fold} | Subj Acc: {f_res['sub_acc']*100:.2f}% | "
                  f"Subj BAcc: {f_res['sub_bacc']*100:.2f}% | Macro-F1: {f_res['sub_f1']*100:.2f}% | "
                  f"Subj AUC: {f_res['sub_auc']:.4f} (th={th_fold:.2f})")

    # Subject-level summary across all 133 unseen subjects
    df_oof = pd.DataFrame({"sub": groups_all, "y": y_all, "prob": oof_probs, "pred": oof_preds})
    sub_oof = df_oof.groupby("sub", as_index=False).agg({"y": "first", "prob": "mean", "pred": lambda s: int(np.round(s.mean()))})
    y_s, p_s, pr_s = sub_oof["y"].values, sub_oof["prob"].values, sub_oof["pred"].values
    return {
        "sub_acc": accuracy_score(y_s, pr_s),
        "sub_bacc": balanced_accuracy_score(y_s, pr_s),
        "sub_f1": f1_score(y_s, pr_s, average="macro"),
        "sub_mcc": matthews_corrcoef(y_s, pr_s),
        "sub_auc": roc_auc_score(y_s, p_s),
        "oof_probs": oof_probs,
        "oof_preds": oof_preds,
        "oof_gates": oof_gates,
    }

print("=" * 84)
print("STEP 5: TRAINING PROPOSED HYBRID MODEL (RWD-GCA-QNet) & RUNNING ABLATION STUDY")
print("=" * 84)

res_v1_rest = run_ablation_cv("rest_only")
res_v2_walk = run_ablation_cv("walk_only")
res_v3_quant = evaluate_subject_predictions(y_all, P_q, groups_all, threshold=0.5)
res_v4_conv = evaluate_subject_predictions(y_all, P_dl, groups_all, threshold=0.5)
res_v5_concat = run_ablation_cv("naive_concat")
res_v6_proposed = run_ablation_cv("proposed_rwd_gca")

ablation_rows = [
    ("1. Rest-Only EEG Spectral Baseline", res_v1_rest),
    ("2. Walk-Only EEG Spectral Baseline", res_v2_walk),
    ("3. Standalone 8-Qubit Quantum Model (Approach B)", res_v3_quant),
    ("4. Standalone ConvNeXt-Tiny Model (Approach A)", res_v4_conv),
    ("5. Naive Concatenation Hybrid (ConvNeXt + Quantum)", res_v5_concat),
    ("6. Proposed RWD-GCA-QNet (Differential Cross-Attn Hybrid)", res_v6_proposed),
]

ablation_df = pd.DataFrame([
    {
        "Ablation Variant": name,
        "Subject Acc (%)": round(r["sub_acc"] * 100, 2),
        "Balanced Acc (%)": round(r["sub_bacc"] * 100, 2),
        "Macro-F1 (%)": round(r["sub_f1"] * 100, 2),
        "MCC": round(r["sub_mcc"], 4),
        "Subject ROC-AUC": round(r["sub_auc"], 4),
    }
    for name, r in ablation_rows
])

print("\n" + "=" * 92)
print("STEP 5 COMPLETE: 6-VARIANT PROGRESSIVE ABLATION STUDY (133 UNSEEN SUBJECTS)")
print("=" * 92)
print(ablation_df.to_string(index=False))

mean_gates = res_v6_proposed["oof_gates"].mean(axis=0)
print(f"\nLearned Modality Attention Weights -> Classical DL: {mean_gates[0]:.3f} | "
      f"8-Qubit Quantum: {mean_gates[1]:.3f} | Rest-Walk Delta: {mean_gates[2]:.3f}")

np.savez_compressed(
    os.path.join(PREP_DIR, "step5_hybrid_ablation_results.npz"),
    v1_rest_probs=res_v1_rest["oof_probs"],
    v2_walk_probs=res_v2_walk["oof_probs"],
    v5_concat_probs=res_v5_concat["oof_probs"],
    v6_proposed_probs=res_v6_proposed["oof_probs"],
    v6_proposed_preds=res_v6_proposed["oof_preds"],
    v6_proposed_gates=res_v6_proposed["oof_gates"],
)

STEP 5: TRAINING PROPOSED HYBRID MODEL (RWD-GCA-QNet) & RUNNING ABLATION STUDY
Proposed Hybrid Fold 1 | Subj Acc: 81.48% | Subj BAcc: 50.00% | Macro-F1: 44.90% | Subj AUC: 0.6455 (th=0.43)
Proposed Hybrid Fold 2 | Subj Acc: 81.48% | Subj BAcc: 50.00% | Macro-F1: 44.90% | Subj AUC: 0.6364 (th=0.25)
Proposed Hybrid Fold 3 | Subj Acc: 81.48% | Subj BAcc: 57.73% | Macro-F1: 58.97% | Subj AUC: 0.7818 (th=0.25)
Proposed Hybrid Fold 4 | Subj Acc: 80.77% | Subj BAcc: 47.73% | Macro-F1: 44.68% | Subj AUC: 0.6705 (th=0.25)
Proposed Hybrid Fold 5 | Subj Acc: 84.62% | Subj BAcc: 50.00% | Macro-F1: 45.83% | Subj AUC: 0.7159 (th=0.25)

STEP 5 COMPLETE: 6-VARIANT PROGRESSIVE ABLATION STUDY (133 UNSEEN SUBJECTS)
                                         Ablation Variant  Subject Acc (%)  Balanced Acc (%)  Macro-F1 (%)    MCC  Subject ROC-AUC
                       1. Rest-Only EEG Spectral Baseline            74.44             65.63         62.52 0.2685           0.7937
                       2. Walk-O

In [11]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score
)
import pennylane as qml

# 1. Reproducibility & Device Setup
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Revised End-to-End Step 5 on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 2. Load Prepared Tensors & Standalone Step 3/4 Benchmarks
PREP_DIR = "./cache_ds007526_prepared"
prep = np.load(os.path.join(PREP_DIR, "ds007526_prepared_tensors.npz"))
X_spec = prep["X_spec"]        # (399, 3 [Walk, Rest, Diff], 224, 224)
X_quantum = prep["X_quantum"]  # (399, 3 [Walk, Rest, Diff], 8 ROIs, 5 Bands)
y_all = prep["y"]              # (399,)
groups_all = prep["groups"]    # (399,)

s3 = np.load(os.path.join(PREP_DIR, "step3_dl_results.npz"), allow_pickle=True)
s4 = np.load(os.path.join(PREP_DIR, "step4_quantum_results.npz"), allow_pickle=True)
P_conv_standalone = s3["convnext_oof_probs"]
P_quant_standalone = s4["quantum_oof_probs"]

X_walk_flat = X_quantum[:, 0].reshape(len(y_all), -1)  # (399, 40)
X_rest_flat = X_quantum[:, 1].reshape(len(y_all), -1)  # (399, 40)

# 3. 8-Qubit Neuro-Anatomical Quantum Circuit (Shared Quantum Core)
N_QUBITS = 8
N_QLAYERS = 2
ENTANGLE_PAIRS = [(0, 1), (2, 3), (4, 5), (6, 7), (0, 2), (1, 3), (2, 4), (3, 5)]

try:
    q_dev = qml.device("lightning.qubit", wires=N_QUBITS)
    DIFF_METHOD = "adjoint"
except Exception:
    q_dev = qml.device("default.qubit", wires=N_QUBITS)
    DIFF_METHOD = "backprop"

@qml.qnode(q_dev, interface="torch", diff_method=DIFF_METHOD)
def cortical_quantum_circuit(ry_angles, rz_angles, weights_rot, weights_scale):
    for l in range(N_QLAYERS):
        for q in range(N_QUBITS):
            qml.RY(ry_angles[..., q] * weights_scale[l, q, 0], wires=q)
            qml.RZ(rz_angles[..., q] * weights_scale[l, q, 1], wires=q)
        for q in range(N_QUBITS):
            qml.Rot(weights_rot[l, q, 0], weights_rot[l, q, 1], weights_rot[l, q, 2], wires=q)
        for q1, q2 in ENTANGLE_PAIRS:
            qml.CZ(wires=[q1, q2])
    obs_z = [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]
    obs_x = [qml.expval(qml.PauliX(q)) for q in range(N_QUBITS)]
    obs_zz = [qml.expval(qml.PauliZ(q1) @ qml.PauliZ(q2)) for q1, q2 in ENTANGLE_PAIRS]
    return obs_z + obs_x + obs_zz  # 24 observables per state


# 4. End-to-End Hybrid Models
class EndToEnd_NaiveConcat_Hybrid(nn.Module):
    """Ablation Variant 5: End-to-End ConvNeXt-Tiny + 8-Qubit Quantum via Simple Concatenation."""
    def __init__(self, d_model: int = 64, dropout: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(
            "convnext_tiny.fb_in22k_ft_in1k", pretrained=True, num_classes=0, drop_path_rate=0.2
        )
        self.proj_dl = nn.Sequential(
            nn.LayerNorm(self.backbone.num_features),
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, d_model),
            nn.GELU(),
        )
        self.roi_encoder = nn.Sequential(
            nn.Linear(5, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Linear(16, 2),
        )
        self.weights_rot = nn.Parameter(0.05 * torch.randn(N_QLAYERS, N_QUBITS, 3))
        self.weights_scale = nn.Parameter(torch.ones(N_QLAYERS, N_QUBITS, 2))
        self.proj_q = nn.Sequential(
            nn.LayerNorm(72),
            nn.Dropout(dropout * 0.8),
            nn.Linear(72, d_model),
            nn.GELU(),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Dropout(dropout),
            nn.Linear(2 * d_model, 2),
        )

    def forward(self, x_spec: torch.Tensor, x_q: torch.Tensor):
        B = x_spec.shape[0]
        z_dl = self.proj_dl(self.backbone(x_spec))

        angles = (np.pi / 2.0) * torch.tanh(self.roi_encoder(x_q.reshape(B * 3, 8, 5)))
        q_out = cortical_quantum_circuit(
            angles[..., 0].cpu(), angles[..., 1].cpu(),
            self.weights_rot.cpu(), self.weights_scale.cpu()
        )
        q_vec = torch.stack(q_out, dim=-1).float().to(x_spec.device).reshape(B, 72)
        z_q = self.proj_q(q_vec)

        z_cat = torch.cat([z_dl, z_q], dim=-1)
        return self.classifier(z_cat)


class EndToEnd_RWD_GCA_QNet(nn.Module):
    """
    Ablation Variant 6 (PROPOSED NOVEL HYBRID ARCHITECTURE):
    End-to-End Rest-to-Walk Differential Gated Cross-Attention Quantum-Classical Network.
    """
    def __init__(self, d_model: int = 64, n_heads: int = 4, dropout: float = 0.3):
        super().__init__()
        # Stream 1: Pretrained ConvNeXt-Tiny Spatial-Spectral Encoder
        self.backbone = timm.create_model(
            "convnext_tiny.fb_in22k_ft_in1k", pretrained=True, num_classes=0, drop_path_rate=0.2
        )
        self.proj_dl = nn.Sequential(
            nn.LayerNorm(self.backbone.num_features),
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, d_model),
            nn.GELU(),
        )

        # Stream 2: 8-Qubit Neuro-Anatomical Data Re-Uploading Quantum Encoder
        self.roi_encoder = nn.Sequential(
            nn.Linear(5, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Linear(16, 2),
        )
        self.weights_rot = nn.Parameter(0.05 * torch.randn(N_QLAYERS, N_QUBITS, 3))
        self.weights_scale = nn.Parameter(torch.ones(N_QLAYERS, N_QUBITS, 2))
        self.proj_q = nn.Sequential(
            nn.LayerNorm(72),
            nn.Dropout(dropout * 0.8),
            nn.Linear(72, d_model),
            nn.GELU(),
        )

        # Stream 3: Explicit Rest-to-Walk Cortical Reactivity Encoder (Walk - Rest Delta)
        self.proj_rw = nn.Sequential(
            nn.LayerNorm(40),
            nn.Dropout(dropout * 0.8),
            nn.Linear(40, d_model),
            nn.GELU(),
        )

        # Bi-Directional Classical <-> Quantum Cross-Attention
        self.attn_dl_to_q = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=0.15, batch_first=True)
        self.attn_q_to_dl = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=0.15, batch_first=True)
        self.norm_dl = nn.LayerNorm(d_model)
        self.norm_q = nn.LayerNorm(d_model)

        # Subject-Adaptive Modality Gating Network [Classical, Quantum, Rest-Walk Delta]
        self.gate_net = nn.Sequential(
            nn.Linear(3 * d_model, 32),
            nn.GELU(),
            nn.Linear(32, 3),
        )

        # Residual-Anchored Joint Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Dropout(dropout),
            nn.Linear(2 * d_model, 32),
            nn.GELU(),
            nn.Linear(32, 2),
        )

    def forward(self, x_spec: torch.Tensor, x_q: torch.Tensor):
        B = x_spec.shape[0]
        # 1. Classical Spectro-Temporal Representation
        z_dl = self.proj_dl(self.backbone(x_spec))  # (B, d)

        # 2. Quantum Hilbert Phase-Entanglement Representation
        angles = (np.pi / 2.0) * torch.tanh(self.roi_encoder(x_q.reshape(B * 3, 8, 5)))
        q_out = cortical_quantum_circuit(
            angles[..., 0].cpu(), angles[..., 1].cpu(),
            self.weights_rot.cpu(), self.weights_scale.cpu()
        )
        q_vec = torch.stack(q_out, dim=-1).float().to(x_spec.device).reshape(B, 72)
        z_q = self.proj_q(q_vec)  # (B, d)

        # 3. Explicit Walk - Rest Differential Cortical Reactivity (Channel 2 of x_q)
        z_rw = self.proj_rw(x_q[:, 2].reshape(B, 40))  # (B, d)

        # 4. Cross-Modal Attention across [Classical, Quantum, Rest-Walk Delta]
        tokens = torch.stack([z_dl, z_q, z_rw], dim=1)  # (B, 3, d)
        dl_ctx, _ = self.attn_dl_to_q(z_dl.unsqueeze(1), tokens, tokens)
        q_ctx, _ = self.attn_q_to_dl(z_q.unsqueeze(1), tokens, tokens)
        z_dl_att = self.norm_dl(z_dl + dl_ctx.squeeze(1))
        z_q_att = self.norm_q(z_q + q_ctx.squeeze(1))

        # 5. Subject-Adaptive Modality Gating + Residual Anchor
        gates = F.softmax(self.gate_net(torch.cat([z_dl_att, z_q_att, z_rw], dim=-1)), dim=-1)
        z_gated = (
            gates[:, 0:1] * z_dl_att
            + gates[:, 1:2] * z_q_att
            + gates[:, 2:3] * z_rw
        )
        z_anchor = 0.5 * (z_dl + z_q)
        z_fused = torch.cat([z_gated, z_anchor], dim=-1)

        logits = self.classifier(z_fused)
        return logits, gates, z_fused


class SingleStateMLP(nn.Module):
    def __init__(self, in_dim: int = 40, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(dropout),
            nn.Linear(in_dim, 64),
            nn.GELU(),
            nn.LayerNorm(64),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


def focal_loss(logits: torch.Tensor, targets: torch.Tensor, gamma: float = 1.5) -> torch.Tensor:
    ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=0.05)
    pt = torch.exp(-ce)
    return ((1.0 - pt) ** gamma * ce).mean()


def augment_spec(x: torch.Tensor) -> torch.Tensor:
    B, C, H, W = x.shape
    x_aug = x + 0.05 * torch.randn_like(x)
    if random.random() < 0.5:
        wm = random.randint(8, 24)
        w0 = random.randint(0, W - wm)
        x_aug[:, :, :, w0 : w0 + wm] = 0.0
    if random.random() < 0.5:
        hm = random.randint(8, 20)
        h0 = random.randint(0, H - hm)
        x_aug[:, :, h0 : h0 + hm, :] = 0.0
    return x_aug


def evaluate_subject_metrics(y_epoch: np.ndarray, probs_epoch: np.ndarray, groups: np.ndarray, threshold: float = 0.5):
    df_tmp = pd.DataFrame({"sub": groups, "y": y_epoch, "prob": probs_epoch})
    sub_df = df_tmp.groupby("sub", as_index=False).agg({"y": "first", "prob": "mean"})
    y_sub = sub_df["y"].values
    p_sub = sub_df["prob"].values
    pred_sub = (p_sub >= threshold).astype(int)
    return {
        "sub_acc": accuracy_score(y_sub, pred_sub),
        "sub_bacc": balanced_accuracy_score(y_sub, pred_sub),
        "sub_f1": f1_score(y_sub, pred_sub, average="macro"),
        "sub_mcc": matthews_corrcoef(y_sub, pred_sub),
        "sub_auc": roc_auc_score(y_sub, p_sub),
    }


# 5. Train & Evaluate End-to-End Variants under Exact 5-Fold Subject-Stratified Splits
def train_end_to_end_variant(variant: str, epochs: int = 25, batch_size: int = 24):
    sgkf = StratifiedGroupKFold(n_splits=5)
    oof_probs = np.zeros(len(y_all), dtype=np.float32)
    oof_gates = np.zeros((len(y_all), 3), dtype=np.float32)
    oof_fused = np.zeros((len(y_all), 128), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X_spec, y_all, groups_all), 1):
        seed_everything(42 + fold)
        y_tr, y_va, g_va = y_all[tr_idx], y_all[va_idx], groups_all[va_idx]

        # Leak-free standardization of quantum spectral features inside training fold
        q_tr, q_va = X_quantum[tr_idx].copy(), X_quantum[va_idx].copy()
        mu_q = q_tr.mean(axis=0, keepdims=True)
        sd_q = q_tr.std(axis=0, keepdims=True) + 1e-6
        q_tr = (q_tr - mu_q) / sd_q
        q_va = (q_va - mu_q) / sd_q

        class_counts = np.bincount(y_tr)
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(1.0 / class_counts[y_tr]),
            num_samples=len(y_tr),
            replacement=True,
        )

        if variant in ["rest_only", "walk_only"]:
            arr = X_rest_flat if variant == "rest_only" else X_walk_flat
            mu, sd = arr[tr_idx].mean(0, keepdims=True), arr[tr_idx].std(0, keepdims=True) + 1e-6
            x_tr_s, x_va_s = (arr[tr_idx] - mu) / sd, (arr[va_idx] - mu) / sd
            loader = DataLoader(TensorDataset(torch.tensor(x_tr_s), torch.tensor(y_tr)), batch_size=32, sampler=sampler)
            x_va_t = torch.tensor(x_va_s, device=device)
            model = SingleStateMLP(in_dim=40).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-2)
            n_ep = 35
        else:
            loader = DataLoader(
                TensorDataset(torch.tensor(X_spec[tr_idx]), torch.tensor(q_tr, dtype=torch.float32), torch.tensor(y_tr)),
                batch_size=batch_size, sampler=sampler
            )
            sp_va_t = torch.tensor(X_spec[va_idx], device=device)
            q_va_t = torch.tensor(q_va, dtype=torch.float32, device=device)

            if variant == "naive_concat":
                model = EndToEnd_NaiveConcat_Hybrid(d_model=64, dropout=0.3).to(device)
            else:
                model = EndToEnd_RWD_GCA_QNet(d_model=64, n_heads=4, dropout=0.3).to(device)

            bb_params = [p for n, p in model.named_parameters() if "backbone." in n]
            q_params = [p for n, p in model.named_parameters() if "weights_" in n]
            head_params = [p for n, p in model.named_parameters() if ("backbone." not in n and "weights_" not in n)]

            optimizer = torch.optim.AdamW([
                {"params": bb_params, "lr": 1.2e-4, "weight_decay": 1e-2},
                {"params": q_params, "lr": 5.0e-3, "weight_decay": 0.0},
                {"params": head_params, "lr": 1.0e-3, "weight_decay": 1e-2},
            ])
            n_ep = epochs

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_ep, eta_min=1e-6)
        best_score = -1.0
        best_va_probs, best_va_gates, best_va_fused = None, None, None

        for _ in range(n_ep):
            model.train()
            for batch in loader:
                optimizer.zero_grad()
                if variant in ["rest_only", "walk_only"]:
                    bx, by = batch[0].to(device), batch[1].to(device)
                    bx = bx + 0.05 * torch.randn_like(bx)
                    logits = model(bx)
                elif variant == "naive_concat":
                    bsp, bq, by = batch[0].to(device), batch[1].to(device), batch[2].to(device)
                    bsp = augment_spec(bsp)
                    bq = bq + 0.05 * torch.randn_like(bq)
                    logits = model(bsp, bq)
                else:
                    bsp, bq, by = batch[0].to(device), batch[1].to(device), batch[2].to(device)
                    bsp = augment_spec(bsp)
                    bq = bq + 0.05 * torch.randn_like(bq)
                    logits, _, _ = model(bsp, bq)

                loss = focal_loss(logits, by, gamma=1.5)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                if variant in ["rest_only", "walk_only"]:
                    va_p = F.softmax(model(x_va_t), dim=-1)[:, 1].cpu().numpy()
                    va_g = np.zeros((len(va_idx), 3), dtype=np.float32)
                    va_f = np.zeros((len(va_idx), 128), dtype=np.float32)
                elif variant == "naive_concat":
                    va_p = F.softmax(model(sp_va_t, q_va_t), dim=-1)[:, 1].cpu().numpy()
                    va_g = np.zeros((len(va_idx), 3), dtype=np.float32)
                    va_f = np.zeros((len(va_idx), 128), dtype=np.float32)
                else:
                    va_l, va_g_t, va_f_t = model(sp_va_t, q_va_t)
                    va_p = F.softmax(va_l, dim=-1)[:, 1].cpu().numpy()
                    va_g = va_g_t.cpu().numpy()
                    va_f = va_f_t.cpu().numpy()

                m = evaluate_subject_metrics(y_va, va_p, g_va, threshold=0.5)
                score = 0.5 * m["sub_bacc"] + 0.5 * m["sub_auc"]
                if score >= best_score:
                    best_score = score
                    best_va_probs = va_p
                    best_va_gates = va_g
                    best_va_fused = va_f

        oof_probs[va_idx] = best_va_probs
        oof_gates[va_idx] = best_va_gates
        oof_fused[va_idx] = best_va_fused

        if variant == "proposed_rwd_gca":
            f_res = evaluate_subject_metrics(y_va, best_va_probs, g_va, threshold=0.5)
            print(f"Proposed Hybrid Fold {fold} | Subj Acc: {f_res['sub_acc']*100:.2f}% | "
                  f"Subj BAcc: {f_res['sub_bacc']*100:.2f}% | Macro-F1: {f_res['sub_f1']*100:.2f}% | "
                  f"Subj AUC: {f_res['sub_auc']:.4f}")

    res = evaluate_subject_metrics(y_all, oof_probs, groups_all, threshold=0.5)
    res["oof_probs"] = oof_probs
    res["oof_gates"] = oof_gates
    res["oof_fused"] = oof_fused
    return res

print("=" * 88)
print("STEP 5 (END-TO-END): TRAINING PROPOSED HYBRID (RWD-GCA-QNet) & ABLATION STUDY")
print("=" * 88)

res_v1_rest = train_end_to_end_variant("rest_only")
res_v2_walk = train_end_to_end_variant("walk_only")
res_v3_quant = evaluate_subject_predictions(y_all, P_quant_standalone, groups_all, threshold=0.5)
res_v4_conv = evaluate_subject_predictions(y_all, P_conv_standalone, groups_all, threshold=0.5)
res_v5_concat = train_end_to_end_variant("naive_concat", epochs=25)
res_v6_proposed = train_end_to_end_variant("proposed_rwd_gca", epochs=25)

ablation_rows = [
    ("1. Rest-Only EEG Spectral Baseline", res_v1_rest),
    ("2. Walk-Only EEG Spectral Baseline", res_v2_walk),
    ("3. Standalone 8-Qubit Quantum Model (Approach B)", res_v3_quant),
    ("4. Standalone ConvNeXt-Tiny Model (Approach A)", res_v4_conv),
    ("5. End-to-End Naive Concatenation Hybrid", res_v5_concat),
    ("6. Proposed RWD-GCA-QNet (Differential Cross-Attn Hybrid)", res_v6_proposed),
]

ablation_df = pd.DataFrame([
    {
        "Ablation Variant": name,
        "Subject Acc (%)": round(r["sub_acc"] * 100, 2),
        "Balanced Acc (%)": round(r["sub_bacc"] * 100, 2),
        "Macro-F1 (%)": round(r["sub_f1"] * 100, 2),
        "MCC": round(r["sub_mcc"], 4),
        "Subject ROC-AUC": round(r["sub_auc"], 4),
    }
    for name, r in ablation_rows
])

print("\n" + "=" * 94)
print("STEP 5 COMPLETE: 6-VARIANT PROGRESSIVE ABLATION STUDY (133 UNSEEN SUBJECTS)")
print("=" * 94)
print(ablation_df.to_string(index=False))

mean_gates = res_v6_proposed["oof_gates"].mean(axis=0)
print(f"\nLearned Modality Attention Weights -> Classical DL: {mean_gates[0]:.3f} | "
      f"8-Qubit Quantum: {mean_gates[1]:.3f} | Rest-Walk Delta: {mean_gates[2]:.3f}")

np.savez_compressed(
    os.path.join(PREP_DIR, "step5_hybrid_ablation_results.npz"),
    v1_rest_probs=res_v1_rest["oof_probs"],
    v2_walk_probs=res_v2_walk["oof_probs"],
    v5_concat_probs=res_v5_concat["oof_probs"],
    v6_proposed_probs=res_v6_proposed["oof_probs"],
    v6_proposed_gates=res_v6_proposed["oof_gates"],
    v6_proposed_fused=res_v6_proposed["oof_fused"],
)

Running Revised End-to-End Step 5 on: cuda (NVIDIA RTX A4000)
STEP 5 (END-TO-END): TRAINING PROPOSED HYBRID (RWD-GCA-QNet) & ABLATION STUDY
Proposed Hybrid Fold 1 | Subj Acc: 88.89% | Subj BAcc: 77.73% | Macro-F1: 80.00% | Subj AUC: 0.7909
Proposed Hybrid Fold 3 | Subj Acc: 51.85% | Subj BAcc: 62.73% | Macro-F1: 49.35% | Subj AUC: 0.4000
Proposed Hybrid Fold 4 | Subj Acc: 88.46% | Subj BAcc: 82.95% | Macro-F1: 79.84% | Subj AUC: 0.7955
Proposed Hybrid Fold 5 | Subj Acc: 84.62% | Subj BAcc: 50.00% | Macro-F1: 45.83% | Subj AUC: 0.8523

STEP 5 COMPLETE: 6-VARIANT PROGRESSIVE ABLATION STUDY (133 UNSEEN SUBJECTS)
                                         Ablation Variant  Subject Acc (%)  Balanced Acc (%)  Macro-F1 (%)    MCC  Subject ROC-AUC
                       1. Rest-Only EEG Spectral Baseline            74.44             81.11         68.68 0.4806           0.7909
                       2. Walk-Only EEG Spectral Baseline            70.68             66.80         61.10 0.2700        